In [ ]:
from google.colab import drive

# This will prompt you to authenticate and authorize Colab to access your Google Drive.
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Data loading and processing

In [ ]:
import torch
import torchvision
import torch.nn as nn
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchvision.transforms import v2

transformations = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomAffine(20),
    v2.ToTensor(),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_test_transformations = v2.Compose([
    v2.ToTensor(),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_folder_plain_small = ImageFolder('/content/drive/MyDrive/Colab Notebooks/DAT341_assignment5/a5_data_small_sample/train', transform=transformations)
train_loader_plain_small =  DataLoader(train_folder_plain_small, batch_size=64, shuffle=True, pin_memory=True, num_workers=2)

val_folder_plain_small = ImageFolder('/content/drive/MyDrive/Colab Notebooks/DAT341_assignment5/a5_data_small_sample/val', transform=val_test_transformations)
val_loader_plain_small = DataLoader(val_folder_plain_small, batch_size=64, pin_memory=True, num_workers=2)


train_folder_plain = ImageFolder('/content/drive/MyDrive/Colab Notebooks/DAT341_assignment5/a5_data/train', transform = torchvision.transforms.ToTensor())
train_loader_plain =  DataLoader(train_folder_plain, batch_size=128, shuffle=True, pin_memory=True, num_workers=2)

val_folder_plain = ImageFolder('/content/drive/MyDrive/Colab Notebooks/DAT341_assignment5/a5_data/val', transform=torchvision.transforms.ToTensor())
val_loader_plain = DataLoader(val_folder_plain, batch_size=128, pin_memory=True, num_workers=2)

train_folder_aug = ImageFolder('/content/drive/MyDrive/Colab Notebooks/DAT341_assignment5/a5_data/train', transform = transformations)
train_loader_aug =  DataLoader(train_folder_aug, batch_size=128, shuffle=True, pin_memory=True, num_workers=2)

val_folder_aug = ImageFolder('/content/drive/MyDrive/Colab Notebooks/DAT341_assignment5/a5_data/val', transform = val_test_transformations)
val_loader_aug = DataLoader(val_folder_aug, batch_size=128, pin_memory=True, num_workers=2)

test_folder = ImageFolder('/content/drive/MyDrive/Colab Notebooks/DAT341_assignment5/a5_data_test/test', transform=val_test_transformations) #not sure wether or not to normalize the train data
test_loader = DataLoader(test_folder, batch_size=128, pin_memory=True, num_workers=2)


/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/_deprecated.py:43: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.
  warnings.warn(


# Training method


In [ ]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_classifier(model, train_data, val_data, hyperparams):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams['lr'])
    loss_func = torch.nn.BCEWithLogitsLoss()
    # Some statistics.
    acc_history = []


    for epoch in range(hyperparams['n_epochs']):

        # Set the model in training mode, enabling dropout if we use that.
        model.train()

        loss_sum = 0

        # For each batch
        for Xbatch, Ybatch in tqdm(train_data):
            optimizer.zero_grad()

            Xbatch = Xbatch.to(device)
            Ybatch = Ybatch.to(device)

            outputs = model(Xbatch)
            outputs = outputs.view(-1)

            loss = loss_func(outputs, Ybatch.float())

            # Update the model.
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()

        mean_loss = loss_sum / len(train_data)

        # Set the model in evaluation mode. Disables dropout if present.
        model.eval()
        with torch.no_grad():
            # Compute the accuracy on the validation data.
            val_acc = predict_and_evaluate(model, val_data)

        acc_history.append(val_acc)

        print(f'Epoch {epoch+1}: loss = {mean_loss:.4f}, val acc = {val_acc:.4f}')

    return acc_history


# A utility function to compute accuracies during training.
def predict_and_evaluate(model, data):
    all_gold = []
    all_pred = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for Xbatch, Ybatch in data:
        Xbatch = Xbatch.to(device)

        outputs = model(Xbatch).view(-1)
        predictions = torch.tensor([1 if output > 0.5 else 0 for output in outputs], dtype=torch.float32)

        all_pred.extend(predictions.cpu().numpy())
        all_gold.extend(Ybatch.numpy())

    return accuracy_score(all_gold, all_pred)

In [ ]:
#for safety, store accuracies here:
accuracies = []

# Baseline model

In [ ]:
cnn_model = nn.Sequential(

            # Block One
            nn.Conv2d(3, 64, kernel_size=5),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Block Two
            nn.Conv2d(64, 32, kernel_size=5),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Block Three
            nn.Conv2d(32, 16, kernel_size=5),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Block Four
            nn.Conv2d(16, 8, kernel_size=3),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Head
            nn.Flatten(),
            nn.Linear(200, 32),
            nn.LeakyReLU(0.01),

            # Output
            nn.Linear(32, 1),
            nn.Sigmoid()

        )

model = cnn_model
history = train_classifier(cnn_model, train_loader_plain, val_loader_plain, {'lr': 1e-4, 'n_epochs': 10})

with torch.no_grad():
  test_acc = predict_and_evaluate(model, test_loader)
  print(f'Test Accuracy Baseline: {test_acc:.4f}')

accuracies.append(test_acc)

# **Baseline with Batch Normalization added**

In [ ]:
cnn_model = nn.Sequential(

            # Block One
            nn.Conv2d(3, 64, kernel_size=5),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Block Two
            nn.Conv2d(64, 32, kernel_size=5),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Block Three
            nn.Conv2d(32, 16, kernel_size=5),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Block Four
            nn.Conv2d(16, 8, kernel_size=3),
            nn.BatchNorm2d(8),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Head
            nn.Flatten(),
            nn.Linear(200, 32),
            nn.LeakyReLU(0.01),

            # Output
            nn.Linear(32, 1),
            nn.Sigmoid()

        )

model = cnn_model
history = train_classifier(cnn_model, train_loader_plain, val_loader_plain, {'lr': 1e-4, 'n_epochs': 10})

with torch.no_grad():
  test_acc = predict_and_evaluate(model, test_loader)
  print(f'Test Accuracy Batch Normalization: {test_acc:.4f}')

accuracies.append(test_acc)

# **Baseline with Transformations added**

In [ ]:
#The only difference is the data

cnn_model = nn.Sequential(

            # Block One
            nn.Conv2d(3, 64, kernel_size=5),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Block Two
            nn.Conv2d(64, 32, kernel_size=5),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Block Three
            nn.Conv2d(32, 16, kernel_size=5),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Block Four
            nn.Conv2d(16, 8, kernel_size=3),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=2),

            # Head
            nn.Flatten(),
            nn.Linear(200, 32),
            nn.LeakyReLU(0.01),

            # Output
            nn.Linear(32, 1),
            nn.Sigmoid()

        )

model = cnn_model
history = train_classifier(cnn_model, train_loader_aug, val_loader_aug, {'lr': 1e-4, 'n_epochs': 10})

with torch.no_grad():
  test_acc = predict_and_evaluate(model, test_loader)
  print(f'Test Accuracy Transformations: {test_acc:.4f}')

accuracies.append(test_acc)

# **Baseline with residual connections**

In [ ]:
import torch.nn as nn

#Here the residuals are reshaped by using a 1x1 convolution for adjusting the number of channels, so that input and output size does not always have to be the same

class CNNModelResidual(nn.Module):
    def __init__(self):
        super(CNNModelResidual, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=5, padding = 2)
        self.relu1 = nn.LeakyReLU(0.01)
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Two
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding = 2)
        self.relu2 = nn.LeakyReLU(0.01)
        self.maxpool2 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Three
        self.conv3 = nn.Conv2d(32, 16, kernel_size=5, padding = 2)
        self.relu3 = nn.LeakyReLU(0.01)
        self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Four
        self.conv4 = nn.Conv2d(16, 8, kernel_size=3, padding = 1)
        self.relu4 = nn.LeakyReLU(0.01)
        self.maxpool4 = nn.MaxPool2d(kernel_size=1)

        # Head
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(131072, 32)
        self.relu5 = nn.LeakyReLU(0.01)

        # Output
        self.linear2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

        self.conv_adjust_channels1 = nn.Conv2d(3, 64, kernel_size=1)
        self.conv_adjust_channels2 = nn.Conv2d(64, 32, kernel_size=1)
        self.conv_adjust_channels3 = nn.Conv2d(32, 16, kernel_size=1)
        self.conv_adjust_channels4 = nn.Conv2d(16, 8, kernel_size=1)



    def forward(self, x):
        # Block One
        residual1 = x
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)

        residual1 = self.conv_adjust_channels1(residual1)
        x = residual1 + x

        residual2 = x

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.maxpool2(x)

        residual2 = self.conv_adjust_channels2(residual2)
        x = x + residual2
        residual3 = x

        x = self.conv3(x)
        x = self.relu3(x)
        x = self.maxpool3(x)

        residual3 = self.conv_adjust_channels3(residual3)

        x = x + residual3
        residual4 = x

        x = self.conv4(x)
        x = self.relu4(x)
        x = self.maxpool4(x)

        residual4 = self.conv_adjust_channels4(residual4)

        x = x + residual4

        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu5(x)


        x = self.linear2(x)
        x = self.sigmoid(x)

        return x

model = CNNModelResidual()

result = train_classifier(model, train_loader_plain, val_loader_plain, {'lr': 1e-4, 'n_epochs': 10})

with torch.no_grad():
  test_acc = predict_and_evaluate(model, test_loader)
  print(f'Test Accuracy Residuals reshaped: {test_acc:.4f}')

accuracies.append(test_acc)

In [ ]:
import torch.nn as nn
#Here output and input size is the same in each convolutional layer

class CNNModelResidual(nn.Module):
    def __init__(self):
        super(CNNModelResidual, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=5, padding = 2)
        self.relu1 = nn.LeakyReLU(0.01)
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Two
        self.conv2 = nn.Conv2d(64, 64, kernel_size=5, padding = 2)
        self.relu2 = nn.LeakyReLU(0.01)
        self.maxpool2 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Three
        self.conv3 = nn.Conv2d(64, 64, kernel_size=5, padding = 2)
        self.relu3 = nn.LeakyReLU(0.01)
        self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Four
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding = 1)
        self.relu4 = nn.LeakyReLU(0.01)
        self.maxpool4 = nn.MaxPool2d(kernel_size=1)

        # Head
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(1048576, 32)
        self.relu5 = nn.LeakyReLU(0.01)

        # Output
        self.linear2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

        self.conv_adjust_channels1 = nn.Conv2d(3, 64, kernel_size=1)


    def forward(self, x):
        # Block One
        residual1 = self.conv_adjust_channels1(x)
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)

        x = residual1 + x
        residual2 = x

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.maxpool2(x)

        x = x + residual2
        residual3 = x

        x = self.conv3(x)
        x = self.relu3(x)
        x = self.maxpool3(x)

        x = x + residual3
        residual4 = x

        x = self.conv4(x)
        x = self.relu4(x)
        x = self.maxpool4(x)

        x = x + residual4

        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu5(x)


        x = self.linear2(x)
        x = self.sigmoid(x)

        return x


model = CNNModelResidual()

result = train_classifier(model, train_loader_plain, val_loader_plain, {'lr': 1e-4, 'n_epochs': 10})

with torch.no_grad():
  test_acc = predict_and_evaluate(model, test_loader)
  print(f'Test Accuracy Residuals same size: {test_acc:.4f}')

accuracies.append(test_acc)


#**Combination of batch normalization, transformation and residual connections**

**First, only batch normalization and residual blocks:**

In [ ]:
import torch.nn as nn

class CNNModelResidual(nn.Module):
    def __init__(self):
        super(CNNModelResidual, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=5, padding = 2)
        self.batch1 = nn.BatchNorm2d(64)
        self.relu1 = nn.LeakyReLU(0.01)
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Two
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding = 2)
        self.batch2 = nn.BatchNorm2d(32)
        self.relu2 = nn.LeakyReLU(0.01)
        self.maxpool2 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Three
        self.conv3 = nn.Conv2d(32, 16, kernel_size=5, padding = 2)
        self.batch3 = nn.BatchNorm2d(16)
        self.relu3 = nn.LeakyReLU(0.01)
        self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Four
        self.conv4 = nn.Conv2d(16, 8, kernel_size=3, padding = 1)
        self.batch4 = nn.BatchNorm2d(8)
        self.relu4 = nn.LeakyReLU(0.01)
        self.maxpool4 = nn.MaxPool2d(kernel_size=1)

        # Head

        #self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(131072, 32)
        self.relu5 = nn.LeakyReLU(0.01)

        # Output
        self.linear2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

        self.conv_adjust_channels1 = nn.Conv2d(3, 64, kernel_size=1)
        self.conv_adjust_channels2 = nn.Conv2d(64, 32, kernel_size=1)
        self.conv_adjust_channels3 = nn.Conv2d(32, 16, kernel_size=1)
        self.conv_adjust_channels4 = nn.Conv2d(16, 8, kernel_size=1)


    def forward(self, x):
        # Block One
        residual1 = x
        x = self.conv1(x)
        x = self.batch1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)

        residual1 = self.conv_adjust_channels1(residual1)
        x = residual1 + x

        residual2 = x

        x = self.conv2(x)
        x = self.batch2(x)
        x = self.relu2(x)
        x = self.maxpool2(x)

        residual2 = self.conv_adjust_channels2(residual2)
        x = x + residual2
        residual3 = x

        x = self.conv3(x)
        x = self.batch3(x)
        x = self.relu3(x)
        x = self.maxpool3(x)

        residual3 = self.conv_adjust_channels3(residual3)

        x = x + residual3
        residual4 = x

        x = self.conv4(x)
        x = self.batch4(x)
        x = self.relu4(x)
        x = self.maxpool4(x)

        residual4 = self.conv_adjust_channels4(residual4)

        x = x + residual4

        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu5(x)


        x = self.linear2(x)
        x = self.sigmoid(x)

        return x


model = CNNModelResidual()
result = train_classifier(model, train_loader_plain, val_loader_plain, {'lr': 1e-4, 'n_epochs': 10})


with torch.no_grad():
  test_acc = predict_and_evaluate(model, test_loader)
  print(f'Test Accuracy Residual blocks and batch norm: {test_acc:.4f}')

accuracies.append(test_acc)


**Second, transformations, batch normalization and residual connections:**

In [ ]:
import torch.nn as nn

class CNNModelResidual(nn.Module):
    def __init__(self):
        super(CNNModelResidual, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=5, padding = 2)
        self.batch1 = nn.BatchNorm2d(64)
        self.relu1 = nn.LeakyReLU(0.01)
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Two
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding = 2)
        self.batch2 = nn.BatchNorm2d(32)
        self.relu2 = nn.LeakyReLU(0.01)
        self.maxpool2 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Three
        self.conv3 = nn.Conv2d(32, 16, kernel_size=5, padding = 2)
        self.batch3 = nn.BatchNorm2d(16)
        self.relu3 = nn.LeakyReLU(0.01)
        self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride = 1, padding = 1)

        # Block Four
        self.conv4 = nn.Conv2d(16, 8, kernel_size=3, padding = 1)
        self.batch4 = nn.BatchNorm2d(8)
        self.relu4 = nn.LeakyReLU(0.01)
        self.maxpool4 = nn.MaxPool2d(kernel_size=1)

        # Head

        #self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(131072, 32)
        self.relu5 = nn.LeakyReLU(0.01)

        # Output
        self.linear2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

        self.conv_adjust_channels1 = nn.Conv2d(3, 64, kernel_size=1)
        self.conv_adjust_channels2 = nn.Conv2d(64, 32, kernel_size=1)
        self.conv_adjust_channels3 = nn.Conv2d(32, 16, kernel_size=1)
        self.conv_adjust_channels4 = nn.Conv2d(16, 8, kernel_size=1)



    def forward(self, x):
        # Block One
        residual1 = x
        x = self.conv1(x)
        x = self.batch1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)

        residual1 = self.conv_adjust_channels1(residual1)
        x = residual1 + x

        residual2 = x

        x = self.conv2(x)
        x = self.batch2(x)
        x = self.relu2(x)
        x = self.maxpool2(x)

        residual2 = self.conv_adjust_channels2(residual2)
        x = x + residual2
        residual3 = x

        x = self.conv3(x)
        x = self.batch3(x)
        x = self.relu3(x)
        x = self.maxpool3(x)

        residual3 = self.conv_adjust_channels3(residual3)

        x = x + residual3
        residual4 = x

        x = self.conv4(x)
        x = self.batch4(x)
        x = self.relu4(x)
        x = self.maxpool4(x)

        residual4 = self.conv_adjust_channels4(residual4)

        x = x + residual4

        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu5(x)


        x = self.linear2(x)
        x = self.sigmoid(x)

        return x


model = CNNModelResidual()
result = train_classifier(model, train_loader_aug, val_loader_aug, {'lr': 1e-4, 'n_epochs': 10})


with torch.no_grad():
  test_acc = predict_and_evaluate(model, test_loader)
  print(f'Test Accuracy Residual blocks, batch norm and transformations: {test_acc:.4f}')


accuracies.append(test_acc)

# Transfer Learning

## ConvNeXt

In [ ]:
import torch
import torchvision
import torch.nn as nn
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as v2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import pandas as pd
from datetime import datetime

training_data_path = Path("a5_data/val")
validation_data_path = Path("a5_data/val")

from tqdm import tqdm
from sklearn.metrics import accuracy_score

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_classifier(model, train_data, val_data, hyperparams):
    model.to(device)

    # Deals with model updates. Adam is more effective than SGD.
    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams['lr'])

    # Cross-entropy loss because we have 2 classes.
    # Note that the softmax is "baked into" this loss, so we should not
    # use a softmax at the end.
    loss_func = torch.nn.BCEWithLogitsLoss()

    # Some statistics (per-epoch)
    acc_history_val = []
    loss_history_train = []
    loss_history_val = []

    best_acc = 0.0
    best_model = None

    for epoch in range(hyperparams['n_epochs']):

        # Set the model in training mode, enabling dropout if we use that.
        model.train()

        loss_sum = 0

        # For each batch
        for Xbatch, Ybatch in tqdm(train_data):
            optimizer.zero_grad()

            Xbatch = Xbatch.to(device)
            Ybatch = Ybatch.to(device)

            outputs = model(Xbatch)
            outputs = outputs.view(-1)

            # Apply the BCE with logits loss.
            loss = loss_func(outputs, Ybatch.float())

            # Update the model.
            loss.backward()
            optimizer.step()

            loss_sum += loss.item()

        mean_loss = loss_sum / len(train_data)
        loss_history_train.append(mean_loss)

        # Set the model in evaluation mode. Disables dropout if present.
        model.eval()
        with torch.no_grad():
            # Compute the accuracy on the validation data.
            val_acc, val_loss = predict_and_evaluate(model, val_data, loss_func)
            if val_acc > best_acc:
                print("New best model")
                best_model = model.state_dict()
                best_acc = val_acc

        loss_history_val.append(val_loss)
        acc_history_val.append(val_acc)

        print(f'Epoch {epoch+1}: train loss = {mean_loss:.4f}, val acc = {val_acc:.4f}, val loss = {val_loss:.4f}')

    # save best here
    fname = f"convnext_{best_acc:.4f}".replace('.','-')
    torch.save(best_model, Path(f"/home/us158207/a5_data/{fname}.pth"))
    pd.DataFrame(acc_history_val).to_csv(Path(f"/home/us158207/a5_data/{fname}_acc_val.csv"))
    pd.DataFrame(loss_history_train).to_csv(Path(f"/home/us158207/a5_data/{fname}_loss_train.csv"))
    pd.DataFrame(loss_history_val).to_csv(Path(f"/home/us158207/a5_data/{fname}_loss_val.csv"))
    return acc_history_val, loss_history_train, loss_history_val


# A utility function to compute accuracies during training.
def predict_and_evaluate(model, data, loss_fn):
    all_gold = []
    all_pred = []
    all_out = []


    for Xbatch, Ybatch in data:
        Xbatch = Xbatch.to(device)
        outputs = model(Xbatch).view(-1)
        all_out.extend(outputs.cpu().numpy())
        # outputs = torch.sigmoid(outputs)
        # predictions = outputs >= 0.5

        # either we check outputs >= 0.5 if we apply the sigmoid, or we check outputs >= 0.0 if we don't (equivalent)
        predictions = outputs >= 0.0

        all_gold.extend(Ybatch.numpy())
        all_pred.extend(predictions.cpu().numpy())

    return accuracy_score(all_gold, all_pred), loss_fn(torch.tensor(all_out).float(), torch.tensor(all_gold).float())


transforms = ConvNeXt_Base_Weights.DEFAULT.transforms()
train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomAffine(20),
    v2.ToTensor(),
    transforms
])

train_folder = ImageFolder(training_data_path, transform=train_transforms)
train_loader = DataLoader(train_folder, batch_size=128, shuffle=True, pin_memory=True, num_workers=2)

val_folder = ImageFolder(validation_data_path, transform=transforms)
val_loader = DataLoader(val_folder, batch_size=128, pin_memory=True, num_workers=2)

convnext = convnext_base(weights=ConvNeXt_Base_Weights.DEFAULT)
# Replace output layer for 2 class classifier
convnext.classifier[2] = nn.Linear(in_features=1024, out_features=1)

# only last layer - 1024 parameters
# update_param_names = ["classifier.2.weight", "classifier.2.bias"]

update_param_names = ["classifier.2.weight", "classifier.2.bias", "classifier.0.weight", "classifier.0.bias"]

# for name, param in convnext.named_parameters():
#     print(name)

for name, param in convnext.named_parameters():
    if name in update_param_names:
        param.requires_grad = True
    else:
        param.requires_grad = False

# print(convnext)
print(f"Trainable parameters: {sum(p.numel() for p in convnext.parameters() if p.requires_grad)}")

acc, loss_train, loss_val = train_classifier(convnext, train_loader, val_loader, {'lr': 1e-3, 'n_epochs': 25})

import matplotlib.pyplot as plt
plt.plot(loss_train,label="train")
plt.plot(loss_val,label="val")
plt.ylim((0,1))
plt.legend()

## VGG16

In [ ]:
import torch
import torchvision
import torch.nn as nn
from torchvision.models import vgg16_bn, VGG16_BN_Weights
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as v2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import pandas as pd
from datetime import datetime

training_data_path = Path("a5_data/train")
validation_data_path = Path("a5_data/val")

from tqdm import tqdm
from sklearn.metrics import accuracy_score

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_classifier(model, train_data, val_data, hyperparams):
    model.to(device)

    # Deals with model updates. Adam is more effective than SGD.
    optimizer = torch.optim.Adam(model.classifier.parameters(), lr=hyperparams['lr'])

    # Cross-entropy loss because we have 2 classes.
    # Note that the softmax is "baked into" this loss, so we should not
    # use a softmax at the end.
    loss_func = torch.nn.BCEWithLogitsLoss()

    # Some statistics (per-epoch)
    acc_history_val = []
    loss_history_train = []
    loss_history_val = []

    best_acc = 0.0
    best_model = None

    for epoch in range(hyperparams['n_epochs']):

        # Set the model in training mode, enabling dropout if we use that.
        model.train()

        loss_sum = 0

        # For each batch
        for Xbatch, Ybatch in tqdm(train_data):
            optimizer.zero_grad()

            Xbatch = Xbatch.to(device)
            Ybatch = Ybatch.to(device)

            outputs = model(Xbatch)
            outputs = outputs.view(-1)

            # Apply the BCE with logits loss.
            loss = loss_func(outputs, Ybatch.float())

            # Update the model.
            loss.backward()
            optimizer.step()

            loss_sum += loss.item()

        mean_loss = loss_sum / len(train_data)
        loss_history_train.append(mean_loss)

        # Set the model in evaluation mode. Disables dropout if present.
        model.eval()
        with torch.no_grad():
            # Compute the accuracy on the validation data.
            val_acc, val_loss = predict_and_evaluate(model, val_data, loss_func)
            if val_acc > best_acc:
                print("New best model")
                best_model = model.state_dict()
                best_acc = val_acc

        loss_history_val.append(val_loss)
        acc_history_val.append(val_acc)

        print(f'Epoch {epoch+1}: train loss = {mean_loss:.4f}, val acc = {val_acc:.4f}, val loss = {val_loss:.4f}')

    # save best here
    fname = f"vgg_{best_acc:.4f}".replace('.','-')
    torch.save(best_model, Path(f"/home/us158207/a5_data/{fname}.pth"))
    pd.DataFrame(acc_history_val).to_csv(Path(f"/home/us158207/a5_data/{fname}_acc_val.csv"))
    pd.DataFrame(loss_history_train).to_csv(Path(f"/home/us158207/a5_data/{fname}_loss_train.csv"))
    pd.DataFrame(loss_history_val).to_csv(Path(f"/home/us158207/a5_data/{fname}_loss_val.csv"))
    return acc_history_val, loss_history_train, loss_history_val


# A utility function to compute accuracies during training.
def predict_and_evaluate(model, data, loss_fn):
    all_gold = []
    all_pred = []
    all_out = []


    for Xbatch, Ybatch in data:
        Xbatch = Xbatch.to(device)
        outputs = model(Xbatch).view(-1)
        all_out.extend(outputs.cpu().numpy())
        # outputs = torch.sigmoid(outputs)
        # predictions = outputs >= 0.5

        # either we check outputs >= 0.5 if we apply the sigmoid, or we check outputs >= 0.0 if we don't (equivalent)
        predictions = outputs >= 0.0

        all_gold.extend(Ybatch.numpy())
        all_pred.extend(predictions.cpu().numpy())

    return accuracy_score(all_gold, all_pred), loss_fn(torch.tensor(all_out).float(), torch.tensor(all_gold).float())


transforms = VGG16_BN_Weights.DEFAULT.transforms()
train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomAffine(20),
    v2.ToTensor(),
    transforms
])

train_folder = ImageFolder(training_data_path, transform=train_transforms)
train_loader = DataLoader(train_folder, batch_size=128, shuffle=True, pin_memory=True, num_workers=2)

val_folder = ImageFolder(validation_data_path, transform=transforms)
val_loader = DataLoader(val_folder, batch_size=128, pin_memory=True, num_workers=2)

vgg = vgg16_bn(weights=VGG16_BN_Weights.DEFAULT)

# Replace output layer for 2 class classifier
vgg.classifier[6] = nn.Linear(in_features=4096, out_features=1)

# only last layer - 4096 parameters
update_param_names = ["classifier.6.weight", "classifier.6.bias"]

# all ~120M parameters of the classifier
# update_param_names = ["classifier.6.weight", "classifier.6.bias", "classifier.3.weight", "classifier.3.bias", "classifier.0.weight", "classifier.0.bias"]

# last two layers: ~17M parameters
# update_param_names = ["classifier.6.weight", "classifier.6.bias", "classifier.3.weight", "classifier.3.bias"]

# for name, param in vgg.named_parameters():
#     print(name)

for name, param in vgg.named_parameters():
    if name in update_param_names:
        param.requires_grad = True
    else:
        param.requires_grad = False

# del vgg.classifier
print(vgg)
print(f"Trainable parameters: {sum(p.numel() for p in vgg.parameters() if p.requires_grad)}")

acc, loss_train, loss_val = train_classifier(vgg, train_loader, val_loader, {'lr': 1e-4, 'n_epochs': 50})

import matplotlib.pyplot as plt
plt.plot(loss_train,label="train")
plt.plot(loss_val,label="val")
plt.ylim((0,1))
plt.legend()

## Test set evaluation

Includes lumped code for all different models evaluated.

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import vgg16_bn, VGG16_BN_Weights
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as v2
import torchvision
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = vgg16_bn()
model.classifier[6] = nn.Linear(in_features=4096, out_features=1)
model.load_state_dict(torch.load(Path("vgg.pth")))
model.to(device)
model.eval()
transforms = VGG16_BN_Weights.DEFAULT.transforms()
test_data_path = Path("a5_data/test")
test_folder = ImageFolder(test_data_path, transform=transforms)
test_dl_full = DataLoader(test_folder, batch_size=128, num_workers=2)

model = convnext_base()
model.classifier[2] = nn.Linear(in_features=1024, out_features=1)
model.load_state_dict(torch.load(Path("convnext.pth")))
model.to(device)
model.eval()
transforms = ConvNeXt_Base_Weights.DEFAULT.transforms()
test_data_path = Path("a5_data/test")
test_folder = ImageFolder(test_data_path, transform=transforms)
test_dl_full = DataLoader(test_folder, batch_size=128, num_workers=2)

model = CNNModelResidual()
model.load_state_dict(torch.load(Path("custom.pth")))
model.to(device)
model.eval()
transforms = v2.Compose([
    v2.ToTensor(),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
test_data_path = Path("a5_data/test")
test_folder = ImageFolder(test_data_path, transform=transforms)
test_dl_full = DataLoader(test_folder, batch_size=128, num_workers=2)


with torch.no_grad():
    all_preds = torch.Tensor().to(device)
    all_gold = torch.Tensor()
    for Xbatch, Ybatch in test_dl_full:
        Xbatch = Xbatch.to(device)
        out = model(Xbatch)
        # the output is the output of the last linear layer
        # we could apply sigmoid and check whether the sigmoid is >= 0.5
        # but we can also directly check the linear output!
        pred = out >= 0.0
        all_preds = torch.cat([all_preds, pred], dim=0)
        all_gold = torch.cat([all_gold, Ybatch], dim=0)

print(accuracy_score(all_preds.cpu(), all_gold))
df = pd.DataFrame(all_preds.cpu())
df = df.applymap(lambda x: "NV" if x else "MEL")
df.to_csv(Path("./test_eval.txt"), index=False, header=False)

## Feature extraction (example)

In [ ]:
import torch
import torchvision
import torch.nn as nn
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as v2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import pandas as pd
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

training_data_path = Path("a5_data/train")
validation_data_path = Path("a5_data/val")
test_data_path = Path("a5_data/test")

transforms = ConvNeXt_Base_Weights.DEFAULT.transforms()
train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomAffine(20),
    v2.ToTensor(),
    transforms
])

train_folder = ImageFolder(training_data_path, transform=train_transforms)
train_loader = DataLoader(train_folder, batch_size=128, shuffle=True, pin_memory=True, num_workers=2)

val_folder = ImageFolder(validation_data_path, transform=transforms)
val_loader = DataLoader(val_folder, batch_size=128, pin_memory=True, num_workers=2)

test_folder = ImageFolder(test_data_path, transform=transforms)
test_loader = DataLoader(test_folder, batch_size=128, pin_memory=True, num_workers=2)

model = convnext_base(weights=ConvNeXt_Base_Weights.DEFAULT)
model = model.features.to(device) # only feature extraction!

# train, validation, test
data = {}

mapping = {'train': {'dl': train_loader, 'x': 'Xtrain', 'y': 'Ytrain'},
           'val': {'dl': val_loader, 'x': 'Xval', 'y': 'Yval'},
           'test': {'dl': test_loader, 'x': 'Xtest', 'y': 'Ytest'}}
with torch.no_grad(): # very important, otherwise will 100% run out of memory!
    for stage in mapping:
        X = torch.Tensor().to(device)
        Y = torch.Tensor()
        for Xbatch, Ybatch in mapping[stage]['dl']:
            Xbatch = Xbatch.to(device)
            X = torch.cat([X, model(Xbatch)], dim=0)
            Y = torch.cat([Y, Ybatch], dim=0)

        data[mapping[stage]['x']] = X.cpu()
        data[mapping[stage]['y']] = Y

torch.save(data, 'extracted_features.pth')